# TOML - Rust

All 9 Rust examples from [docs/toml.md](https://platob.github.io/yggdryl/toml/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::{Value, toml};

let source = "title = \"yggdryl\"\ncount = 3\n\n[owner]\nname = \"Ada\"\n";
let value = toml::from_str(source)?;

assert_eq!(value.get_key_str("title"), Some(&Value::from("yggdryl")));
assert_eq!(value.get_key_str("count"), Some(&Value::I64(3)));
let owner = value.get_key_str("owner").unwrap();
assert_eq!(owner.get_key_str("name"), Some(&Value::from("Ada")));

assert_eq!(toml::from_slice(&toml::to_vec(&value)?)?, value);

## Table order

In [ ]:
use yggdryl::{Value, toml};

let value = Value::from_mapping([
    (Value::from("zeta"), Value::I64(1)),
    (Value::from("beta"), Value::I64(2)),
    (
        Value::from("alpha"),
        Value::from_mapping([(Value::from("deep"), Value::I64(3))])?,
    ),
])?;

let encoded = toml::to_vec(&value)?;
assert_eq!(
    String::from_utf8(encoded.clone())?,
    "\"zeta\" = 1\n\"beta\" = 2\n\"alpha\" = {\"deep\" = 3}\n",
);

let decoded = toml::from_slice(&encoded)?;
assert_eq!(decoded, value);
let keys: Vec<&str> = decoded
    .mapping_iter()
    .map(|(key, _)| key.as_str().unwrap())
    .collect();
assert_eq!(keys, ["zeta", "beta", "alpha"]);

## The type mapping

In [ ]:
use yggdryl::{Value, toml};

let value = toml::from_str(concat!(
    "text = \"café\"\n",
    "integer = 7\n",
    "hex = 0x2a\n",
    "float = 1.5\n",
    "infinite = inf\n",
    "negative_zero = -0.0\n",
    "flag = true\n",
    "array = [1, \"two\"]\n",
    "table = { nested = 1 }\n",
    "moment = 1979-05-27T07:32:00Z\n",
))?;

assert_eq!(value.get_key_str("text").and_then(Value::as_str), Some("café"));
assert_eq!(value.get_key_str("integer"), Some(&Value::I64(7)));
assert_eq!(value.get_key_str("hex"), Some(&Value::I64(42)));
assert_eq!(value.get_key_str("float").and_then(Value::as_f64), Some(1.5));
assert_eq!(
    value.get_key_str("infinite").and_then(Value::as_f64),
    Some(f64::INFINITY),
);
assert_eq!(
    value
        .get_key_str("negative_zero")
        .and_then(Value::as_f64)
        .map(f64::to_bits),
    Some((-0.0_f64).to_bits()),
);
assert_eq!(value.get_key_str("flag"), Some(&Value::Bool(true)));
assert_eq!(value.get_key_str("array").map(Value::len), Some(2));
assert_eq!(
    value.get_key_str("table").unwrap().get_key_str("nested"),
    Some(&Value::I64(1)),
);
assert_eq!(value.get_key_str("moment").unwrap().kind(), "timestamp");

## Values TOML has no syntax for

In [ ]:
use yggdryl::{Value, toml};

let value = Value::from_mapping([
    (Value::from("missing"), Value::Null),
    (Value::from("blob"), Value::from(vec![0_u8, 255])),
    (Value::from("huge"), Value::U128(u128::MAX)),
])?;

let encoded = toml::to_vec(&value)?;
assert!(
    String::from_utf8(encoded.clone())?
        .contains("\"missing\" = { \"$yggdryl\" = { version = 1, type = \"null\" } }")
);
assert_eq!(toml::from_slice(&encoded)?, value);

// A TOML root is a table, so a non-table root is wrapped the same way.
let root = Value::from("scalar root");
assert_eq!(toml::from_slice(&toml::to_vec(&root)?)?, root);

// A user table that only looks like an envelope stays user data.
let lookalike = Value::from_mapping([(
    Value::from("$yggdryl"),
    Value::from_mapping([
        (Value::from("version"), Value::I64(1)),
        (Value::from("type"), Value::from("null")),
    ])?,
)])?;
assert_eq!(toml::from_slice(&toml::to_vec(&lookalike)?)?, lookalike);

## Dates and times

In [ ]:
use yggdryl::{TimeUnit, Timezone, Value, toml};

let value = toml::from_str(concat!(
    "offset = 1979-05-27T07:32:00Z\n",
    "local = 1979-05-27T07:32:00\n",
    "day = 1979-05-27\n",
    "clock = 07:32:00\n",
))?;

// An offset reading is an instant: the count is UTC and the zone is the offset.
assert_eq!(
    value.get_key_str("offset"),
    Some(&Value::timestamp_in(296_638_320, TimeUnit::Second, Some(Timezone::UTC))),
);
// A local reading carries no zone, which is how a naive reading is spelled.
assert_eq!(
    value.get_key_str("local"),
    Some(&Value::timestamp_in(296_638_320, TimeUnit::Second, None)),
);
assert_eq!(value.get_key_str("day"), Some(&Value::date(3_433)));
assert_eq!(
    value.get_key_str("clock"),
    Some(&Value::time(27_120, TimeUnit::Second)),
);

// Each form goes back out in the syntax it arrived in.
let encoded = String::from_utf8(toml::to_vec(&value)?)?;
assert!(encoded.contains("\"offset\" = 1979-05-27T07:32:00Z\n"));
assert!(encoded.contains("\"day\" = 1979-05-27\n"));

// A zone that names a place is not an offset, so it spells the classic
// string - offset and bracketed name - instead of being rewritten as the
// offset that place happens to be at.
let paris = Value::from_mapping([(
    Value::from("at"),
    Value::timestamp(296_638_320, TimeUnit::Second, Some("Europe/Paris"))?,
)])?;
let encoded = String::from_utf8(toml::to_vec(&paris)?)?;
assert!(encoded.contains(r#""at" = "1979-05-27T09:32:00+02:00[Europe/Paris]""#));

## Exactly one document

In [ ]:
use std::io::Cursor;

use yggdryl::{Value, toml};

// The root is a table, so an empty or comment-only document is an empty table.
let empty = Value::from_mapping([])?;
assert_eq!(toml::from_str("# nothing to see\n")?, empty);
assert!(toml::to_vec(&empty)?.is_empty());

// The reader is an iterator that yields exactly one document.
let mut reader = toml::Reader::new(Cursor::new(b"id = 1"));
assert!(reader.next().unwrap().is_ok());
assert!(reader.next().is_none());

// `to_writer_all` rejects zero and two before it writes a byte.
let one = Value::from_mapping([(Value::from("id"), Value::I64(1))])?;
let mut output = Vec::new();
assert!(toml::to_writer_all(&mut output, [one.clone(), one.clone()]).is_err());
assert!(output.is_empty());
toml::to_writer_all(&mut output, std::iter::once(&one))?;
assert_eq!(toml::from_slice_all(&output)?, vec![one]);

## Laying out a dump

In [ ]:
use yggdryl::generic::Value;
use yggdryl::text::Formatting;

let value = Value::from_mapping([
    (Value::String("id".into()), Value::I64(1)),
    (Value::String("tags".into()), Value::from_sequence([Value::String("a".into())])),
])?;

assert_eq!(yggdryl::toml::to_vec(&value)?, b"\"id\" = 1\n\"tags\" = [\"a\"]\n");
assert_eq!(
    yggdryl::toml::to_vec_with_formatting(&value, Formatting::indented(2))?,
    b"\"id\" = 1\n\"tags\" = [\n  \"a\",\n]\n",
);

// Formatting changes bytes, never meaning.
assert_eq!(
    yggdryl::toml::from_slice(
        &yggdryl::toml::to_vec_with_formatting(&value, Formatting::indented(2))?,
    )?,
    value,
);

## Failures

In [ ]:
use yggdryl::{Error, Limits, Value, toml};

let source = "ok = 0\nnested = { a = 1, a = 2 }\n";
match toml::from_str(source).unwrap_err() {
    Error::Codec {
        format,
        position,
        reason,
    } => {
        assert_eq!(format, "toml");
        assert_eq!(position, source.rfind("a = 2").unwrap());
        assert!(reason.contains("duplicate"));
    }
    other => panic!("unexpected error: {other}"),
}

assert!(toml::from_str("big = 9223372036854775808").is_err());

// Depth is measured on the wire projection, where a mapping TOML cannot key
// costs four containers: the wrapper table, the body, the entry array, and
// the pair array.
let arbitrary = |depth: usize| {
    (0..depth).fold(Value::from("payload"), |value, _| {
        Value::from_mapping([(Value::Bool(true), value)]).expect("one key")
    })
};
let budget = Limits::new(48, 1024, 1024, 1);
assert!(toml::validate_for_write_with_limits(&arbitrary(12), budget).is_ok());
assert!(toml::validate_for_write_with_limits(&arbitrary(13), budget).is_err());

// A temporal TOML spells itself is a leaf, so it costs no container at all.
let day = Value::from_mapping([(Value::from("day"), Value::date(3_433))])?;
assert!(toml::validate_for_write_with_limits(&day, Limits::new(1, 1024, 1024, 1)).is_ok());

// A decimal costs the envelope table, its body, and the array that body holds.
let price = Value::from_mapping([(Value::from("price"), Value::decimal(125, 2))])?;
assert!(toml::validate_for_write_with_limits(&price, Limits::new(3, 1024, 1024, 1)).is_err());
assert!(toml::validate_for_write_with_limits(&price, Limits::new(4, 1024, 1024, 1)).is_ok());

// That check runs before anything is written.
assert_eq!(toml::MAX_PARSER_DEPTH, 64);
let mut output = Vec::new();
assert!(toml::to_writer(&mut output, &arbitrary(64)).is_err());
assert!(output.is_empty());

## Placeholders

In [ ]:
use yggdryl::text::{Format, Loading, Placeholders};
use yggdryl::Value;

let placeholders = Placeholders::new()
    .with_variable("HOST", Value::from("db.internal"))
    .with_variable("PORT", Value::I64(5432));
let loading = Loading::new().with_placeholders(placeholders);

let document = "[database]\nhost = \"{{ HOST }}\"\nport = \"{{ PORT }}\"\n";
let value = yggdryl::text::from_str_with(document, Format::Toml, &loading)?;
let database = value.get_key_str("database").expect("the table");
assert_eq!(database.get_key_str("host").and_then(Value::as_str), Some("db.internal"));
assert_eq!(database.get_key_str("port"), Some(&Value::I64(5432)));